In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from tqdm.auto import tqdm
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs, prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel
from pd_estim_A.models.nig.nig_apath import (
    NIGParams,
    build_weekly_calendar_from_panel,
    solve_esscher_theta,
    invert_assets_weekly_for_firm,
)
from pd_estim_A.models.nig.nig_em import (
    em_fit_nig,
    fit_nig_params_from_weekly_assets,
    project_delta_to_sample_var,
    project_mu_to_sample_mean,
)
from pd_estim_A.models.nig.nig_pd import pd_terminal_nig_weekly

# Rolling settings
TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"
HORIZON_WEEKS = 52.0
T_INV = 1.0
DATA_END = pd.Timestamp("2024-12-31")
LAST_TRAIN_END = DATA_END - pd.offsets.QuarterEnd(1)

# Feasibility / regularization
ALPHA_SEED_MIN = 2.25
ALPHA_MIN_FINAL = 2.25
ESSCHER_MARGIN = 0.25

# Numerics
INVERT_U = 120.0
INVERT_N = 2000
EM_MAX_ITER = 80
EM_TOL = 1e-6

# Data filters
MIN_DAILY_ROWS = 200
MIN_WEEKLY_RETURNS = 60

c:\Users\vkeenan\AppData\Local\miniconda3\envs\Accenture\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
    gvkey       date             E          isin  \
0  100022 2012-01-03  3.328431e+10  DE0005190003   
1  100080 2012-01-03  4.268705e+10  DE000BAY0017   
2  100312 2012-01-03  1.469717e+09  DE0007030009   
3  100581 2012-01-03  4.935351e+10  FR0000120321   
4  100957 2012-01-03  2.931

In [3]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
nig = nig_df.copy()
nig["gvkey"] = nig["gvkey"].astype(str)
nig["date"]  = pd.to_datetime(nig["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(nig["gvkey"].unique()) & set(cds["gvkey"].unique()))
nig = nig[nig["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = nig["date"].min(), nig["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
nig = nig.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    nig,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

nig_df = merged_cds.reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 76508
date range: 2012-01-03 00:00:00 → 2025-12-19 00:00:00


In [4]:
def nig_weekly_to_annual(p: NIGParams, weeks_per_year: float = 52.0) -> NIGParams:
    return NIGParams(
        alpha=float(p.alpha),
        beta=float(p.beta),
        delta=float(p.delta) * float(weeks_per_year),
        mu=float(p.mu) * float(weeks_per_year),
    )

def nig_annual_to_weekly(p: NIGParams, weeks_per_year: float = 52.0) -> NIGParams:
    return NIGParams(
        alpha=float(p.alpha),
        beta=float(p.beta),
        delta=float(p.delta) / float(weeks_per_year),
        mu=float(p.mu) / float(weeks_per_year),
    )

def row_to_nig_params(row: pd.Series) -> NIGParams:
    return NIGParams(
        alpha=float(row["alpha"]),
        beta=float(row["beta"]),
        delta=float(row["delta"]),
        mu=float(row["mu"]),
    )

def build_proxy_inversion_seed(
    x_proxy: np.ndarray,
    p_proxy_raw: NIGParams,
    *,
    alpha_seed_min: float = 2.25,
    esscher_margin: float = 0.25,
) -> tuple[NIGParams, NIGParams]:
    """
    Build a proxy-informed NIG seed:
      - weekly seed for final EM on implied asset returns
      - annual seed for pricing inversion
    """
    x_proxy = np.asarray(x_proxy, float)
    x_proxy = x_proxy[np.isfinite(x_proxy)]

    sample_mean = float(np.mean(x_proxy))
    sample_var = float(np.var(x_proxy, ddof=1))

    beta_seed = float(p_proxy_raw.beta)
    alpha_seed = max(
        float(p_proxy_raw.alpha),
        float(alpha_seed_min),
        abs(beta_seed) + 1.0 + float(esscher_margin),
    )

    p_tmp = NIGParams(alpha=alpha_seed, beta=beta_seed, delta=1.0, mu=0.0)
    p_tmp = project_delta_to_sample_var(p_tmp, sample_var)
    p_seed_weekly = project_mu_to_sample_mean(p_tmp, sample_mean)
    p_seed_annual = nig_weekly_to_annual(p_seed_weekly)

    return p_seed_weekly, p_seed_annual

def fit_one_nig_window_from_assets(
    weekly_assets: pd.DataFrame,
    p0_weekly: NIGParams,
    *,
    alpha_min: float = 2.25,
    em_max_iter: int = 80,
    em_tol: float = 1e-6,
    min_weekly_returns: int = 60,
) -> tuple[NIGParams, pd.DataFrame]:
    n_ret = int(weekly_assets["dlogA"].notna().sum())
    if n_ret < min_weekly_returns:
        raise ValueError(f"Too few weekly implied asset returns: {n_ret}")

    upd = fit_nig_params_from_weekly_assets(
        weekly_assets,
        p0=p0_weekly,
        window_weeks=n_ret,
        refit_every=n_ret,
        em_max_iter=em_max_iter,
        em_tol=em_tol,
        alpha_min=alpha_min,
        use_theta=True,
        prefer_precomputed_theta=True,
        verbose=False,
    )
    if upd.empty:
        raise RuntimeError("No NIG update row produced.")

    return row_to_nig_params(upd.iloc[-1]), upd

def compute_pd_panel_for_const_params(
    weekly_df: pd.DataFrame,
    p_weekly: NIGParams,
    p_annual: NIGParams,
    *,
    horizon_weeks: float = 52.0,
    tau_inv: float = 1.0,
) -> pd.DataFrame:
    """
    Compute physical PDs always, and risk-neutral PDs when Esscher Q is feasible.
    """
    out = weekly_df.copy().sort_values("date").reset_index(drop=True)

    pd_p = []
    pd_q = []
    theta_q = []
    beta_q = []
    q_feasible = []
    margin_theta = []
    margin_theta1 = []

    for _, row in out.iterrows():
        A = float(row["A_hat"])
        L = float(row["L"])
        r = float(row["r"])

        # Physical PD
        pd_p_val = pd_terminal_nig_weekly(A, L, p_weekly, horizon_weeks=horizon_weeks)
        pd_p.append(float(pd_p_val))

        # Risk-neutral feasibility + PD_Q
        try:
            th = solve_esscher_theta(p_annual, r, tau=tau_inv)
            p_q_weekly = NIGParams(
                alpha=float(p_weekly.alpha),
                beta=float(p_weekly.beta) + float(th),
                delta=float(p_weekly.delta),
                mu=float(p_weekly.mu),
            )
            p_q_weekly.validate()

            pd_q_val = pd_terminal_nig_weekly(A, L, p_q_weekly, horizon_weeks=horizon_weeks)

            q_feasible.append(True)
            theta_q.append(float(th))
            beta_q.append(float(p_q_weekly.beta))
            pd_q.append(float(pd_q_val))
            margin_theta.append(float(p_annual.alpha - abs(p_annual.beta + th)))
            margin_theta1.append(float(p_annual.alpha - abs(p_annual.beta + th + 1.0)))
        except Exception:
            q_feasible.append(False)
            theta_q.append(np.nan)
            beta_q.append(np.nan)
            pd_q.append(np.nan)
            margin_theta.append(np.nan)
            margin_theta1.append(np.nan)

    out["PD_P_1y"] = pd_p
    out["PD_Q_1y"] = pd_q
    out["theta_q"] = theta_q
    out["beta_q"] = beta_q
    out["q_feasible"] = q_feasible
    out["margin_theta"] = margin_theta
    out["margin_theta1"] = margin_theta1
    return out

In [5]:
def nig_weekly_to_annual(p: NIGParams, weeks_per_year: float = 52.0) -> NIGParams:
    return NIGParams(
        alpha=float(p.alpha),
        beta=float(p.beta),
        delta=float(p.delta) * float(weeks_per_year),
        mu=float(p.mu) * float(weeks_per_year),
    )

def nig_annual_to_weekly(p: NIGParams, weeks_per_year: float = 52.0) -> NIGParams:
    return NIGParams(
        alpha=float(p.alpha),
        beta=float(p.beta),
        delta=float(p.delta) / float(weeks_per_year),
        mu=float(p.mu) / float(weeks_per_year),
    )

def row_to_nig_params(row: pd.Series) -> NIGParams:
    return NIGParams(
        alpha=float(row["alpha"]),
        beta=float(row["beta"]),
        delta=float(row["delta"]),
        mu=float(row["mu"]),
    )

def build_proxy_inversion_seed(
    x_proxy: np.ndarray,
    p_proxy_raw: NIGParams,
    *,
    alpha_seed_min: float = 2.25,
    esscher_margin: float = 0.25,
) -> tuple[NIGParams, NIGParams]:
    """
    Build a proxy-informed NIG seed:
      - weekly seed for final EM on implied asset returns
      - annual seed for pricing inversion
    """
    x_proxy = np.asarray(x_proxy, float)
    x_proxy = x_proxy[np.isfinite(x_proxy)]

    sample_mean = float(np.mean(x_proxy))
    sample_var = float(np.var(x_proxy, ddof=1))

    beta_seed = float(p_proxy_raw.beta)
    alpha_seed = max(
        float(p_proxy_raw.alpha),
        float(alpha_seed_min),
        abs(beta_seed) + 1.0 + float(esscher_margin),
    )

    p_tmp = NIGParams(alpha=alpha_seed, beta=beta_seed, delta=1.0, mu=0.0)
    p_tmp = project_delta_to_sample_var(p_tmp, sample_var)
    p_seed_weekly = project_mu_to_sample_mean(p_tmp, sample_mean)
    p_seed_annual = nig_weekly_to_annual(p_seed_weekly)

    return p_seed_weekly, p_seed_annual

def fit_one_nig_window_from_assets(
    weekly_assets: pd.DataFrame,
    p0_weekly: NIGParams,
    *,
    alpha_min: float = 2.25,
    em_max_iter: int = 80,
    em_tol: float = 1e-6,
    min_weekly_returns: int = 60,
) -> tuple[NIGParams, pd.DataFrame]:
    n_ret = int(weekly_assets["dlogA"].notna().sum())
    if n_ret < min_weekly_returns:
        raise ValueError(f"Too few weekly implied asset returns: {n_ret}")

    upd = fit_nig_params_from_weekly_assets(
        weekly_assets,
        p0=p0_weekly,
        window_weeks=n_ret,
        refit_every=n_ret,
        em_max_iter=em_max_iter,
        em_tol=em_tol,
        alpha_min=alpha_min,
        use_theta=True,
        prefer_precomputed_theta=True,
        verbose=False,
    )
    if upd.empty:
        raise RuntimeError("No NIG update row produced.")

    return row_to_nig_params(upd.iloc[-1]), upd

def compute_pd_panel_for_const_params(
    weekly_df: pd.DataFrame,
    p_weekly: NIGParams,
    p_annual: NIGParams,
    *,
    horizon_weeks: float = 52.0,
    tau_inv: float = 1.0,
) -> pd.DataFrame:
    """
    Compute physical PDs always, and risk-neutral PDs when Esscher Q is feasible.
    """
    out = weekly_df.copy().sort_values("date").reset_index(drop=True)

    pd_p = []
    pd_q = []
    theta_q = []
    beta_q = []
    q_feasible = []
    margin_theta = []
    margin_theta1 = []

    for _, row in out.iterrows():
        A = float(row["A_hat"])
        L = float(row["L"])
        r = float(row["r"])

        # Physical PD
        pd_p_val = pd_terminal_nig_weekly(A, L, p_weekly, horizon_weeks=horizon_weeks)
        pd_p.append(float(pd_p_val))

        # Risk-neutral feasibility + PD_Q
        try:
            th = solve_esscher_theta(p_annual, r, tau=tau_inv)
            p_q_weekly = NIGParams(
                alpha=float(p_weekly.alpha),
                beta=float(p_weekly.beta) + float(th),
                delta=float(p_weekly.delta),
                mu=float(p_weekly.mu),
            )
            p_q_weekly.validate()

            pd_q_val = pd_terminal_nig_weekly(A, L, p_q_weekly, horizon_weeks=horizon_weeks)

            q_feasible.append(True)
            theta_q.append(float(th))
            beta_q.append(float(p_q_weekly.beta))
            pd_q.append(float(pd_q_val))
            margin_theta.append(float(p_annual.alpha - abs(p_annual.beta + th)))
            margin_theta1.append(float(p_annual.alpha - abs(p_annual.beta + th + 1.0)))
        except Exception:
            q_feasible.append(False)
            theta_q.append(np.nan)
            beta_q.append(np.nan)
            pd_q.append(np.nan)
            margin_theta.append(np.nan)
            margin_theta1.append(np.nan)

    out["PD_P_1y"] = pd_p
    out["PD_Q_1y"] = pd_q
    out["theta_q"] = theta_q
    out["beta_q"] = beta_q
    out["q_feasible"] = q_feasible
    out["margin_theta"] = margin_theta
    out["margin_theta1"] = margin_theta1
    return out

In [6]:
# Cell 1 — multi-firm panel prep

panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "E", "L", "r", "cds"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["E", "L", "r", "cds"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

firm_daily = {}
firm_meta = {}

for gvkey, g_firm in panel.groupby("gvkey", sort=False):
    g_firm = g_firm.sort_values("date").copy()
    g_firm = g_firm.groupby("date", as_index=False).last()
    firm_daily[gvkey] = g_firm.set_index("date")
    firm_meta[gvkey] = {
        "company": str(g_firm["company"].dropna().iloc[0]) if ("company" in g_firm.columns and g_firm["company"].notna().any()) else "",
    }

gvkeys_all = sorted(firm_daily.keys())

print("Firms loaded:", len(gvkeys_all))
print("Panel date range:", panel["date"].min(), "->", panel["date"].max())
display(panel.head())

Firms loaded: 21
Panel date range: 2012-01-03 00:00:00 -> 2025-12-19 00:00:00


,gvkey,date,company,E,L,r,cds
0,100022,2012-01-03,BAYERISCHE MOTOREN WERKE AKT,3.328431e+10,8.576700e+10,0.001177,NaN
1,100022,2012-01-04,BAYERISCHE MOTOREN WERKE AKT,3.363347e+10,8.576700e+10,0.001037,NaN
2,100022,2012-01-05,BAYERISCHE MOTOREN WERKE AKT,3.380203e+10,8.576700e+10,0.001614,NaN
3,100022,2012-01-06,BAYERISCHE MOTOREN WERKE AKT,3.344083e+10,8.576700e+10,0.001873,NaN
4,100022,2012-01-09,BAYERISCHE MOTOREN WERKE AKT,3.421741e+10,8.576700e+10,0.001835,NaN


In [7]:
# Cell 3 — weekly-aligned rolling window schedule for the full panel

master_weekly = build_weekly_calendar_from_panel(panel[["date"]].drop_duplicates().copy(), week_ending=WEEK_ENDING)

def align_to_prev_week(ts: pd.Timestamp, week_grid: pd.DatetimeIndex) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    pos = week_grid.searchsorted(ts, side="right") - 1
    if pos < 0:
        return pd.NaT
    return pd.Timestamp(week_grid[pos])

def align_to_next_week(ts: pd.Timestamp, week_grid: pd.DatetimeIndex) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    pos = week_grid.searchsorted(ts, side="left")
    if pos >= len(week_grid):
        return pd.NaT
    return pd.Timestamp(week_grid[pos])

global_min_date = panel["date"].min()
earliest_cal_end = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)

train_end_anchors = pd.date_range(start=earliest_cal_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_end_anchors = pd.to_datetime(train_end_anchors)

windows = []
for train_end_anchor in train_end_anchors:
    train_end = align_to_prev_week(train_end_anchor, master_weekly)
    if pd.isna(train_end):
        continue

    raw_train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    train_start = align_to_next_week(raw_train_start, master_weekly)

    next_q_end_anchor = train_end_anchor + pd.offsets.QuarterEnd(1)
    oos_start = align_to_next_week(train_end + pd.Timedelta(days=1), master_weekly)
    oos_end = align_to_prev_week(next_q_end_anchor, master_weekly)

    if pd.isna(train_start) or pd.isna(oos_start) or pd.isna(oos_end):
        continue
    if train_start > train_end or oos_start > oos_end:
        continue

    windows.append(
        {
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
            "train_end_anchor": pd.Timestamp(train_end_anchor),
            "oos_end_anchor": pd.Timestamp(next_q_end_anchor),
        }
    )

windows_df = pd.DataFrame(windows)
display(windows_df.head())
display(windows_df.tail())
print("n_windows:", len(windows_df))

,train_start,train_end,oos_start,oos_end,train_end_anchor,oos_end_anchor
0,2012-03-30,2014-03-28,2014-04-04,2014-06-27,2014-03-31,2014-06-30
1,2012-06-29,2014-06-27,2014-07-04,2014-09-26,2014-06-30,2014-09-30
2,2012-09-28,2014-09-26,2014-10-03,2014-12-26,2014-09-30,2014-12-31
3,2012-12-28,2014-12-26,2015-01-02,2015-03-27,2014-12-31,2015-03-31
4,2013-03-29,2015-03-27,2015-04-03,2015-06-26,2015-03-31,2015-06-30


,train_start,train_end,oos_start,oos_end,train_end_anchor,oos_end_anchor
38,2021-10-01,2023-09-29,2023-10-06,2023-12-29,2023-09-30,2023-12-31
39,2021-12-31,2023-12-29,2024-01-05,2024-03-29,2023-12-31,2024-03-31
40,2022-04-01,2024-03-29,2024-04-05,2024-06-28,2024-03-31,2024-06-30
41,2022-07-01,2024-06-28,2024-07-05,2024-09-27,2024-06-30,2024-09-30
42,2022-09-30,2024-09-27,2024-10-04,2024-12-27,2024-09-30,2024-12-31


n_windows: 43


In [8]:
# Cell 4 — rolling estimation for all firms

roll_summary_rows = []
roll_weekly_is_rows = []
roll_weekly_oos_rows = []

for w in tqdm(windows, desc="Rolling NIG windows"):
    train_start = w["train_start"]
    train_end = w["train_end"]
    oos_start = w["oos_start"]
    oos_end = w["oos_end"]
    train_end_anchor = w["train_end_anchor"]
    oos_end_anchor = w["oos_end_anchor"]

    for gvkey in gvkeys_all:
        g_all = firm_daily.get(gvkey)
        company = firm_meta.get(gvkey, {}).get("company", "")

        if g_all is None or g_all.empty:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": "missing_firm_panel",
                }
            )
            continue

        # -------------------------
        # Training daily slice
        # -------------------------
        g_train = g_all.loc[(g_all.index >= train_start) & (g_all.index <= train_end)].copy()
        g_train = (
            g_train.reset_index()
                   .dropna(subset=["date", "E", "L", "r"])
                   .query("E > 0 and L > 0")
                   .sort_values("date")
                   .reset_index(drop=True)
        )

        if len(g_train) < MIN_DAILY_ROWS:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": "too_few_daily_rows_train",
                    "n_daily_train": int(len(g_train)),
                }
            )
            continue

        # -------------------------
        # Weekly proxy returns on training slice
        # -------------------------
        week_dates_train = build_weekly_calendar_from_panel(g_train[["date"]].copy(), week_ending=WEEK_ENDING)
        g_train_idx = g_train.set_index("date")

        weekly_proxy = (
            g_train_idx.loc[week_dates_train, ["E", "L", "r"]]
                      .copy()
                      .reset_index()
                      .rename(columns={"index": "date"})
        )
        weekly_proxy["logE"] = np.log(weekly_proxy["E"])
        weekly_proxy["dlogE"] = weekly_proxy["logE"].diff()

        x_proxy = weekly_proxy["dlogE"].dropna().to_numpy(float)

        if len(x_proxy) < MIN_WEEKLY_RETURNS:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": f"too_few_weekly_proxy_returns:{len(x_proxy)}",
                    "n_daily_train": int(len(g_train)),
                }
            )
            continue

        # -------------------------
        # Raw proxy fit -> projected weekly/annual seed
        # -------------------------
        proxy_std = float(np.std(x_proxy, ddof=1))
        P0_PROXY_RAW = NIGParams(
            alpha=5.0,
            beta=0.0,
            delta=max(0.10 * proxy_std, 1e-4),
            mu=float(np.mean(x_proxy)),
        )

        try:
            p_proxy_raw = em_fit_nig(
                x_proxy,
                P0_PROXY_RAW,
                max_iter=200,
                tol=1e-6,
                verbose=False,
            )

            p_seed_weekly, p_seed_annual = build_proxy_inversion_seed(
                x_proxy,
                p_proxy_raw,
                alpha_seed_min=ALPHA_SEED_MIN,
                esscher_margin=ESSCHER_MARGIN,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": f"fail_proxy_seed:{type(e).__name__}:{str(e)[:180]}",
                }
            )
            continue

        # -------------------------
        # One inversion pass on training window
        # -------------------------
        try:
            weekly_is = invert_assets_weekly_for_firm(
                g_train[["date", "E", "L", "r"]].copy(),
                p_seed_annual,
                tau=T_INV,
                U=INVERT_U,
                n=INVERT_N,
                week_ending=WEEK_ENDING,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": f"fail_train_inversion:{type(e).__name__}:{str(e)[:180]}",
                }
            )
            continue

        n_weekly_ret = int(weekly_is["dlogA"].notna().sum())
        if n_weekly_ret < MIN_WEEKLY_RETURNS:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": f"too_few_implied_asset_returns:{n_weekly_ret}",
                    "n_weekly_train": int(len(weekly_is)),
                }
            )
            continue

        # -------------------------
        # Final weekly NIG fit on implied asset returns
        # -------------------------
        try:
            p_hat_weekly, upd_one = fit_one_nig_window_from_assets(
                weekly_is,
                p_seed_weekly,
                alpha_min=ALPHA_MIN_FINAL,
                em_max_iter=EM_MAX_ITER,
                em_tol=EM_TOL,
                min_weekly_returns=MIN_WEEKLY_RETURNS,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": f"fail_train_em:{type(e).__name__}:{str(e)[:180]}",
                    "n_weekly_train": int(len(weekly_is)),
                }
            )
            continue

        p_hat_annual = nig_weekly_to_annual(p_hat_weekly)

        # -------------------------
        # In-sample PD panel
        # -------------------------
        pd_is = compute_pd_panel_for_const_params(
            weekly_is,
            p_hat_weekly,
            p_hat_annual,
            horizon_weeks=HORIZON_WEEKS,
            tau_inv=T_INV,
        )

        last_is = pd_is.sort_values("date").iloc[-1]

        roll_summary_rows.append(
            {
                "gvkey": gvkey,
                "company": company,
                "train_start": train_start,
                "train_end": train_end,
                "oos_start": oos_start,
                "oos_end": oos_end,
                "train_end_anchor": train_end_anchor,
                "oos_end_anchor": oos_end_anchor,
                "ok": True,
                "msg": "ok",
                "alpha": float(p_hat_weekly.alpha),
                "beta": float(p_hat_weekly.beta),
                "delta": float(p_hat_weekly.delta),
                "mu": float(p_hat_weekly.mu),
                "alpha_required_from_theta": float(upd_one.iloc[-1]["alpha_required_from_theta"])
                    if np.isfinite(upd_one.iloc[-1]["alpha_required_from_theta"]) else np.nan,
                "alpha_hits_floor": bool(upd_one.iloc[-1]["alpha_hits_floor"]),
                "emp_mean": float(upd_one.iloc[-1]["emp_mean"]),
                "mod_mean": float(upd_one.iloc[-1]["mod_mean"]),
                "emp_std": float(upd_one.iloc[-1]["emp_std"]),
                "mod_std": float(upd_one.iloc[-1]["mod_std"]),
                "pd_date_is": pd.Timestamp(last_is["date"]),
                "A_used_is": float(last_is["A_hat"]),
                "L_used_is": float(last_is["L"]),
                "r_used_is": float(last_is["r"]),
                "PD_P_1y_is": float(last_is["PD_P_1y"]),
                "PD_Q_1y_is": float(last_is["PD_Q_1y"]) if np.isfinite(last_is["PD_Q_1y"]) else np.nan,
                "q_feasible_is": bool(last_is["q_feasible"]),
                "theta_q_is": float(last_is["theta_q"]) if np.isfinite(last_is["theta_q"]) else np.nan,
                "margin_theta_is": float(last_is["margin_theta"]) if np.isfinite(last_is["margin_theta"]) else np.nan,
                "margin_theta1_is": float(last_is["margin_theta1"]) if np.isfinite(last_is["margin_theta1"]) else np.nan,
                "n_daily_train": int(len(g_train)),
                "n_weekly_train": int(len(weekly_is)),
                "n_weekly_returns_train": int(n_weekly_ret),
            }
        )

        weekly_is_store = pd_is.copy()
        weekly_is_store["gvkey"] = gvkey
        weekly_is_store["company"] = company
        weekly_is_store["train_end"] = train_end
        weekly_is_store["train_end_anchor"] = train_end_anchor
        weekly_is_store["oos_end_anchor"] = oos_end_anchor
        weekly_is_store["alpha"] = float(p_hat_weekly.alpha)
        weekly_is_store["beta"] = float(p_hat_weekly.beta)
        weekly_is_store["delta"] = float(p_hat_weekly.delta)
        weekly_is_store["mu"] = float(p_hat_weekly.mu)
        roll_weekly_is_rows.append(weekly_is_store)

        # -------------------------
        # OOS slice: invert under annualized fitted training params
        # -------------------------
        g_oos = g_all.loc[(g_all.index >= oos_start) & (g_all.index <= oos_end)].copy()
        g_oos = (
            g_oos.reset_index()
                 .dropna(subset=["date", "E", "L", "r"])
                 .query("E > 0 and L > 0")
                 .sort_values("date")
                 .reset_index(drop=True)
        )

        if len(g_oos) == 0:
            continue

        try:
            weekly_oos = invert_assets_weekly_for_firm(
                g_oos[["date", "E", "L", "r"]].copy(),
                p_hat_annual,
                tau=T_INV,
                U=INVERT_U,
                n=INVERT_N,
                week_ending=WEEK_ENDING,
            )
        except Exception as e:
            roll_summary_rows.append(
                {
                    "gvkey": gvkey,
                    "company": company,
                    "train_start": train_start,
                    "train_end": train_end,
                    "oos_start": oos_start,
                    "oos_end": oos_end,
                    "train_end_anchor": train_end_anchor,
                    "oos_end_anchor": oos_end_anchor,
                    "ok": False,
                    "msg": f"fail_oos_inversion:{type(e).__name__}:{str(e)[:180]}",
                }
            )
            continue

        pd_oos = compute_pd_panel_for_const_params(
            weekly_oos,
            p_hat_weekly,
            p_hat_annual,
            horizon_weeks=HORIZON_WEEKS,
            tau_inv=T_INV,
        )

        weekly_oos_store = pd_oos.copy()
        weekly_oos_store["gvkey"] = gvkey
        weekly_oos_store["company"] = company
        weekly_oos_store["train_end"] = train_end
        weekly_oos_store["train_end_anchor"] = train_end_anchor
        weekly_oos_store["oos_end_anchor"] = oos_end_anchor
        weekly_oos_store["alpha"] = float(p_hat_weekly.alpha)
        weekly_oos_store["beta"] = float(p_hat_weekly.beta)
        weekly_oos_store["delta"] = float(p_hat_weekly.delta)
        weekly_oos_store["mu"] = float(p_hat_weekly.mu)
        roll_weekly_oos_rows.append(weekly_oos_store)

Rolling NIG windows:  14%|█▍        | 6/43 [12:29<1:35:54, 155.53s/it]c:\Users\vkeenan\AppData\Local\miniconda3\envs\Accenture\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2027: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  return integrate.quad(self._pdf, _a, x, args=args)[0]
Rolling NIG windows:  60%|██████    | 26/43 [1:49:03<1:04:58, 229.31s/it]c:\Users\vkeenan\AppData\Local\miniconda3\envs\Accenture\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2027: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  return integrate.quad(self._pdf, 

In [10]:
# Cell 5 — assemble outputs

roll_summary_nig_df = (
    pd.DataFrame(roll_summary_rows)
      .sort_values(["train_end", "gvkey", "ok"], ascending=[True, True, False])
      .reset_index(drop=True)
)

roll_weekly_nig_is_df = (
    pd.concat(roll_weekly_is_rows, ignore_index=True)
      .sort_values(["train_end", "gvkey", "date"])
      .reset_index(drop=True)
) if len(roll_weekly_is_rows) else pd.DataFrame()

roll_weekly_nig_oos_df = (
    pd.concat(roll_weekly_oos_rows, ignore_index=True)
      .sort_values(["train_end", "gvkey", "date"])
      .reset_index(drop=True)
) if len(roll_weekly_oos_rows) else pd.DataFrame()

print("roll_summary_nig_df shape:", roll_summary_nig_df.shape)
print("roll_weekly_nig_is_df shape:", roll_weekly_nig_is_df.shape)
print("roll_weekly_nig_oos_df shape:", roll_weekly_nig_oos_df.shape)

display(roll_summary_nig_df.head())
display(roll_weekly_nig_is_df.head())
display(roll_weekly_nig_oos_df.head())

roll_summary_nig_df shape: (93, 33)
roll_weekly_nig_is_df shape: (5250, 24)
roll_weekly_nig_oos_df shape: (364, 24)


,gvkey,company,train_start,train_end,oos_start,oos_end,train_end_anchor,oos_end_anchor,ok,msg,...,r_used_is,PD_P_1y_is,PD_Q_1y_is,q_feasible_is,theta_q_is,margin_theta_is,margin_theta1_is,n_daily_train,n_weekly_train,n_weekly_returns_train
0,100022,BAYERISCHE MOTOREN WERKE AKT,2012-03-30,2014-03-28,2014-04-04,2014-06-27,2014-03-31,2014-06-30,True,ok,...,0.001019,0.002153,NaN,False,NaN,NaN,NaN,521.0,105.0,104.0
1,100022,BAYERISCHE MOTOREN WERKE AKT,2012-03-30,2014-03-28,2014-04-04,2014-06-27,2014-03-31,2014-06-30,False,fail_oos_inversion:RuntimeError:Could not brac...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100080,BAYER AG,2012-03-30,2014-03-28,2014-04-04,2014-06-27,2014-03-31,2014-06-30,False,fail_train_inversion:RuntimeError:Could not br...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100957,IBERDROLA SA,2012-03-30,2014-03-28,2014-04-04,2014-06-27,2014-03-31,2014-06-30,True,ok,...,0.001019,0.003410,0.005584,True,-0.684124,1.565127,1.934873,521.0,105.0,104.0
4,101202,L'AIR LIQUIDE SA,2012-03-30,2014-03-28,2014-04-04,2014-06-27,2014-03-31,2014-06-30,True,ok,...,0.001019,0.000247,0.014858,True,-2.180939,0.068805,1.068805,521.0,105.0,104.0


,date,E,L,r,A_hat,theta,logA,dlogA,PD_P_1y,PD_Q_1y,...,margin_theta1,gvkey,company,train_end,train_end_anchor,oos_end_anchor,alpha,beta,delta,mu
0,2012-03-30,4.059254e+10,9.632600e+10,0.001602,1.367644e+11,-2.041635,25.641525,NaN,0.003188,NaN,...,NaN,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2014-03-31,2014-06-30,2.25,0.000275,0.000273,0.001396
1,2012-04-06,4.020124e+10,9.632600e+10,0.001453,1.363874e+11,-2.042250,25.638765,-0.002760,0.003233,NaN,...,NaN,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2014-03-31,2014-06-30,2.25,0.000275,0.000273,0.001396
2,2012-04-13,4.063468e+10,9.632600e+10,0.001169,1.368482e+11,-2.043420,25.642138,0.003373,0.003178,NaN,...,NaN,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2014-03-31,2014-06-30,2.25,0.000275,0.000273,0.001396
3,2012-04-20,4.204334e+10,9.632600e+10,0.001586,1.382167e+11,-2.041700,25.652089,0.009951,0.003022,NaN,...,NaN,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2014-03-31,2014-06-30,2.25,0.000275,0.000273,0.001396
4,2012-04-27,4.342191e+10,9.632600e+10,0.001112,1.396409e+11,-2.043655,25.662340,0.010251,0.002872,NaN,...,NaN,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2014-03-31,2014-06-30,2.25,0.000275,0.000273,0.001396


,date,E,L,r,A_hat,theta,logA,dlogA,PD_P_1y,PD_Q_1y,...,margin_theta1,gvkey,company,train_end,train_end_anchor,oos_end_anchor,alpha,beta,delta,mu
0,2014-04-04,3.246644e+10,5.705039e+10,0.001217,8.944742e+10,-0.659303,25.216917,NaN,0.003413,0.005485,...,1.910052,100957,IBERDROLA SA,2014-03-28,2014-03-31,2014-06-30,2.25,-0.000749,0.000335,0.000048
1,2014-04-11,3.052891e+10,5.705039e+10,0.001092,8.751703e+10,-0.674956,25.195099,-0.021818,0.003803,0.006075,...,1.925705,100957,IBERDROLA SA,2014-03-28,2014-03-31,2014-06-30,2.25,-0.000749,0.000335,0.000048
2,2014-04-18,3.089219e+10,5.705039e+10,0.000972,8.788714e+10,-0.689877,25.199319,0.004220,0.003724,0.006034,...,1.940626,100957,IBERDROLA SA,2014-03-28,2014-03-31,2014-06-30,2.25,-0.000749,0.000335,0.000048
3,2014-04-25,3.137020e+10,5.705039e+10,0.001185,8.835305e+10,-0.663399,25.204607,0.005287,0.003627,0.005789,...,1.914147,100957,IBERDROLA SA,2014-03-28,2014-03-31,2014-06-30,2.25,-0.000749,0.000335,0.000048
4,2014-05-02,3.236447e+10,5.705039e+10,0.000924,8.936217e+10,-0.695885,25.215963,0.011357,0.003429,0.005660,...,1.946634,100957,IBERDROLA SA,2014-03-28,2014-03-31,2014-06-30,2.25,-0.000749,0.000335,0.000048


In [11]:
# Cell 6 — final multi-firm rolling panel

is_last = (
    roll_weekly_nig_is_df.sort_values(["gvkey", "train_end", "date"])
    .groupby(["gvkey", "train_end"], as_index=False)
    .tail(1)
    [["gvkey", "company", "train_end", "train_end_anchor", "oos_end_anchor",
      "date", "A_hat", "L", "PD_P_1y", "PD_Q_1y", "q_feasible",
      "alpha", "beta", "delta", "mu"]]
    .rename(columns={"A_hat": "A_used", "L": "L_used"})
)

train_end_rows = pd.DataFrame(
    {
        "gvkey": is_last["gvkey"].astype(str),
        "company": is_last["company"],
        "date": pd.to_datetime(is_last["date"]),
        "alpha": is_last["alpha"].astype(float),
        "beta": is_last["beta"].astype(float),
        "delta": is_last["delta"].astype(float),
        "mu": is_last["mu"].astype(float),
        "A_used": is_last["A_used"].astype(float),
        "L_used": is_last["L_used"].astype(float),
        "PD_P_1y": is_last["PD_P_1y"].astype(float),
        "PD_Q_1y": is_last["PD_Q_1y"].astype(float),
        "q_feasible": is_last["q_feasible"].astype(bool),
        "train_end_date": pd.to_datetime(is_last["train_end"]),
        "train_end_anchor": pd.to_datetime(is_last["train_end_anchor"]),
        "oos_end_anchor": pd.to_datetime(is_last["oos_end_anchor"]),
        "training_end": 1,
    }
)

oos_rows = pd.DataFrame(
    {
        "gvkey": roll_weekly_nig_oos_df["gvkey"].astype(str),
        "company": roll_weekly_nig_oos_df["company"],
        "date": pd.to_datetime(roll_weekly_nig_oos_df["date"]),
        "alpha": roll_weekly_nig_oos_df["alpha"].astype(float),
        "beta": roll_weekly_nig_oos_df["beta"].astype(float),
        "delta": roll_weekly_nig_oos_df["delta"].astype(float),
        "mu": roll_weekly_nig_oos_df["mu"].astype(float),
        "A_used": roll_weekly_nig_oos_df["A_hat"].astype(float),
        "L_used": roll_weekly_nig_oos_df["L"].astype(float),
        "PD_P_1y": roll_weekly_nig_oos_df["PD_P_1y"].astype(float),
        "PD_Q_1y": roll_weekly_nig_oos_df["PD_Q_1y"].astype(float),
        "q_feasible": roll_weekly_nig_oos_df["q_feasible"].astype(bool),
        "train_end_date": pd.to_datetime(roll_weekly_nig_oos_df["train_end"]),
        "train_end_anchor": pd.to_datetime(roll_weekly_nig_oos_df["train_end_anchor"]),
        "oos_end_anchor": pd.to_datetime(roll_weekly_nig_oos_df["oos_end_anchor"]),
        "training_end": 0,
    }
)

final_nig_roll_all = pd.concat([train_end_rows, oos_rows], ignore_index=True)
final_nig_roll_all = (
    final_nig_roll_all.sort_values(["gvkey", "date", "training_end"], ascending=[True, True, False])
                     .drop_duplicates(subset=["gvkey", "date"], keep="first")
                     .sort_values(["gvkey", "date"])
                     .reset_index(drop=True)
)

display(final_nig_roll_all.head(20))
print(final_nig_roll_all.shape)

,gvkey,company,date,alpha,beta,delta,mu,A_used,L_used,PD_P_1y,PD_Q_1y,q_feasible,train_end_date,train_end_anchor,oos_end_anchor,training_end
0,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2.25,0.000275,0.000273,0.001396,1.581424e+11,1.027250e+11,0.002153,NaN,False,2014-03-28,2014-03-31,2014-06-30,1
1,100022,BAYERISCHE MOTOREN WERKE AKT,2014-06-27,2.25,0.000442,0.000251,0.001857,1.583382e+11,1.027250e+11,0.001763,NaN,False,2014-06-27,2014-06-30,2014-09-30,1
2,100022,BAYERISCHE MOTOREN WERKE AKT,2014-09-26,2.25,0.000319,0.000266,0.001608,1.543773e+11,1.027250e+11,0.002232,NaN,False,2014-09-26,2014-09-30,2014-12-31,1
3,100022,BAYERISCHE MOTOREN WERKE AKT,2014-12-26,2.25,0.000268,0.000274,0.001113,1.574858e+11,1.027250e+11,0.002366,NaN,False,2014-12-26,2014-12-31,2015-03-31,1
4,100957,IBERDROLA SA,2014-03-28,2.25,-0.000749,0.000335,0.000048,8.946513e+10,5.705039e+10,0.003410,0.005584,True,2014-03-28,2014-03-31,2014-06-30,1
5,100957,IBERDROLA SA,2014-04-04,2.25,-0.000749,0.000335,0.000048,8.944742e+10,5.705039e+10,0.003413,0.005485,True,2014-03-28,2014-03-31,2014-06-30,0
6,100957,IBERDROLA SA,2014-04-11,2.25,-0.000749,0.000335,0.000048,8.751703e+10,5.705039e+10,0.003803,0.006075,True,2014-03-28,2014-03-31,2014-06-30,0
7,100957,IBERDROLA SA,2014-04-18,2.25,-0.000749,0.000335,0.000048,8.788714e+10,5.705039e+10,0.003724,0.006034,True,2014-03-28,2014-03-31,2014-06-30,0
8,100957,IBERDROLA SA,2014-04-25,2.25,-0.000749,0.000335,0.000048,8.835305e+10,5.705039e+10,0.003627,0.005789,True,2014-03-28,2014-03-31,2014-06-30,0
9,100957,IBERDROLA SA,2014-05-02,2.25,-0.000749,0.000335,0.000048,8.936217e+10,5.705039e+10,0.003429,0.005660,True,2014-03-28,2014-03-31,2014-06-30,0


(398, 16)


In [12]:
# Cell 6 — final multi-firm rolling panel

is_last = (
    roll_weekly_nig_is_df.sort_values(["gvkey", "train_end", "date"])
    .groupby(["gvkey", "train_end"], as_index=False)
    .tail(1)
    [["gvkey", "company", "train_end", "train_end_anchor", "oos_end_anchor",
      "date", "A_hat", "L", "PD_P_1y", "PD_Q_1y", "q_feasible",
      "alpha", "beta", "delta", "mu"]]
    .rename(columns={"A_hat": "A_used", "L": "L_used"})
)

train_end_rows = pd.DataFrame(
    {
        "gvkey": is_last["gvkey"].astype(str),
        "company": is_last["company"],
        "date": pd.to_datetime(is_last["date"]),
        "alpha": is_last["alpha"].astype(float),
        "beta": is_last["beta"].astype(float),
        "delta": is_last["delta"].astype(float),
        "mu": is_last["mu"].astype(float),
        "A_used": is_last["A_used"].astype(float),
        "L_used": is_last["L_used"].astype(float),
        "PD_P_1y": is_last["PD_P_1y"].astype(float),
        "PD_Q_1y": is_last["PD_Q_1y"].astype(float),
        "q_feasible": is_last["q_feasible"].astype(bool),
        "train_end_date": pd.to_datetime(is_last["train_end"]),
        "train_end_anchor": pd.to_datetime(is_last["train_end_anchor"]),
        "oos_end_anchor": pd.to_datetime(is_last["oos_end_anchor"]),
        "training_end": 1,
    }
)

oos_rows = pd.DataFrame(
    {
        "gvkey": roll_weekly_nig_oos_df["gvkey"].astype(str),
        "company": roll_weekly_nig_oos_df["company"],
        "date": pd.to_datetime(roll_weekly_nig_oos_df["date"]),
        "alpha": roll_weekly_nig_oos_df["alpha"].astype(float),
        "beta": roll_weekly_nig_oos_df["beta"].astype(float),
        "delta": roll_weekly_nig_oos_df["delta"].astype(float),
        "mu": roll_weekly_nig_oos_df["mu"].astype(float),
        "A_used": roll_weekly_nig_oos_df["A_hat"].astype(float),
        "L_used": roll_weekly_nig_oos_df["L"].astype(float),
        "PD_P_1y": roll_weekly_nig_oos_df["PD_P_1y"].astype(float),
        "PD_Q_1y": roll_weekly_nig_oos_df["PD_Q_1y"].astype(float),
        "q_feasible": roll_weekly_nig_oos_df["q_feasible"].astype(bool),
        "train_end_date": pd.to_datetime(roll_weekly_nig_oos_df["train_end"]),
        "train_end_anchor": pd.to_datetime(roll_weekly_nig_oos_df["train_end_anchor"]),
        "oos_end_anchor": pd.to_datetime(roll_weekly_nig_oos_df["oos_end_anchor"]),
        "training_end": 0,
    }
)

final_nig_roll_all = pd.concat([train_end_rows, oos_rows], ignore_index=True)
final_nig_roll_all = (
    final_nig_roll_all.sort_values(["gvkey", "date", "training_end"], ascending=[True, True, False])
                     .drop_duplicates(subset=["gvkey", "date"], keep="first")
                     .sort_values(["gvkey", "date"])
                     .reset_index(drop=True)
)

display(final_nig_roll_all.head(20))
print(final_nig_roll_all.shape)

,gvkey,company,date,alpha,beta,delta,mu,A_used,L_used,PD_P_1y,PD_Q_1y,q_feasible,train_end_date,train_end_anchor,oos_end_anchor,training_end
0,100022,BAYERISCHE MOTOREN WERKE AKT,2014-03-28,2.25,0.000275,0.000273,0.001396,1.581424e+11,1.027250e+11,0.002153,NaN,False,2014-03-28,2014-03-31,2014-06-30,1
1,100022,BAYERISCHE MOTOREN WERKE AKT,2014-06-27,2.25,0.000442,0.000251,0.001857,1.583382e+11,1.027250e+11,0.001763,NaN,False,2014-06-27,2014-06-30,2014-09-30,1
2,100022,BAYERISCHE MOTOREN WERKE AKT,2014-09-26,2.25,0.000319,0.000266,0.001608,1.543773e+11,1.027250e+11,0.002232,NaN,False,2014-09-26,2014-09-30,2014-12-31,1
3,100022,BAYERISCHE MOTOREN WERKE AKT,2014-12-26,2.25,0.000268,0.000274,0.001113,1.574858e+11,1.027250e+11,0.002366,NaN,False,2014-12-26,2014-12-31,2015-03-31,1
4,100957,IBERDROLA SA,2014-03-28,2.25,-0.000749,0.000335,0.000048,8.946513e+10,5.705039e+10,0.003410,0.005584,True,2014-03-28,2014-03-31,2014-06-30,1
5,100957,IBERDROLA SA,2014-04-04,2.25,-0.000749,0.000335,0.000048,8.944742e+10,5.705039e+10,0.003413,0.005485,True,2014-03-28,2014-03-31,2014-06-30,0
6,100957,IBERDROLA SA,2014-04-11,2.25,-0.000749,0.000335,0.000048,8.751703e+10,5.705039e+10,0.003803,0.006075,True,2014-03-28,2014-03-31,2014-06-30,0
7,100957,IBERDROLA SA,2014-04-18,2.25,-0.000749,0.000335,0.000048,8.788714e+10,5.705039e+10,0.003724,0.006034,True,2014-03-28,2014-03-31,2014-06-30,0
8,100957,IBERDROLA SA,2014-04-25,2.25,-0.000749,0.000335,0.000048,8.835305e+10,5.705039e+10,0.003627,0.005789,True,2014-03-28,2014-03-31,2014-06-30,0
9,100957,IBERDROLA SA,2014-05-02,2.25,-0.000749,0.000335,0.000048,8.936217e+10,5.705039e+10,0.003429,0.005660,True,2014-03-28,2014-03-31,2014-06-30,0


(398, 16)


In [13]:
# save final_df as CSV in the current working directory
output_path = Path.cwd() / ".." / "data" / "derived"
final_nig_roll_all.to_csv(output_path / "NIG_weekly_rolling.csv", index=False)